# Compare dense and hybrid retrieval

The next cell finds the repository and loads its settings. The notebook connects to the configured PostgreSQL database and local bge-m3 model, using a dedicated owner.

In [1]:
import os
import sys
from pathlib import Path

working_dir = Path.cwd().resolve()
repo_root = next(
    folder for folder in (working_dir, *working_dir.parents)
    if (folder / "app").is_dir() and (folder / "corpus").is_dir()
)
# Settings resolve .env from the working directory.
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

from app.config import get_settings
from app.db import SessionFactory, engine, init_db
from app.errors import NoExtractableTextError
from app.ingestion.pipeline import ingest_document
from app.llm_client import EmbeddingClient
from app.repository import delete_document
from app.retrieval import count_owner_documents

settings = get_settings()
owner = "notebook"
from app.ingestion.extractors import SUPPORTED_EXTENSIONS
from app.retrieval import dense_search, keyword_search, reciprocal_rank_fusion

corpus = sorted(
    path for path in (repo_root / "corpus").iterdir()
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

## Choose questions and a comparison format

The next cell defines three questions that belong to the corpus, an off-topic control, and a formatter. Each row will show the same cosine score wherever the chunk appears, alongside any keyword rank and fusion score.

In [2]:
questions = [
    "How many annual leave days do employees get?",
    "What expenses are reimbursed when traveling for a client project?",
    "What is the process for reporting a security incident?",
]
off_topic = "What is the capital of France?"


def show_hits(mode, hits):
    print(f"{mode:>6} | No. | Similarity | Keyword rank | RRF score | Source")
    for index, hit in enumerate(hits[:4], 1):
        rank = hit.keyword_rank if hit.keyword_rank is not None else "-"
        source = f"{hit.filename} (page {hit.page or '-'} / {hit.section or '-'})"
        print(f"{mode:>6} | {index:>3} | {hit.similarity:>10.4f} | {rank:>12} | "
              f"{hit.rrf_score:>9.6f} | {source}")

## Index the corpus and run both searches

The next cell uses the ingestion pipeline, then embeds each question once so the two strategies use the same query vector. It prints the four highest-ranked hits for each strategy and the off-topic question’s best dense score. A finally block removes only documents created by this run.

In [3]:
await init_db()
embedder = EmbeddingClient(settings)
created_ids = []
try:
    async with SessionFactory() as session:
        if await count_owner_documents(session, owner):
            raise RuntimeError("The notebook owner already has documents. Use an empty notebook owner.")
        for path in corpus:
            try:
                doc = await ingest_document(
                    path.read_bytes(), path.name, owner, session, embedder, settings
                )
                created_ids.append(doc.id)
            except NoExtractableTextError:
                print(f"Skipped {path.name}: no text layer.")

        print(f"Indexed {len(created_ids)} documents with {settings.embed_model}.")
        vectors = await embedder.embed([*questions, off_topic])
        for question, vector in zip(questions, vectors[:-1], strict=True):
            print(f"\nQuestion: {question}")
            async with session.begin():
                dense = await dense_search(session, owner, vector, settings.candidate_k)
                keyword = await keyword_search(
                    session, owner, question, settings.candidate_k, query_vector=vector
                )
            hybrid = reciprocal_rank_fusion(dense, keyword, settings.rrf_k)
            show_hits("dense", dense)
            show_hits("hybrid", hybrid)
            print("Top-4 dense similarities:", [hit.similarity for hit in dense[:4]])
            print("Top-4 hybrid similarities:", [hit.similarity for hit in hybrid[:4]])

        async with session.begin():
            unrelated = await dense_search(session, owner, vectors[-1], 1)
        print(f"\nOff-topic question: {off_topic}")
        print(f"Off-topic best similarity: {unrelated[0].similarity:.4f}")
        print(f"Configured minimum similarity: {settings.min_similarity:.4f}")
finally:
    try:
        async with SessionFactory() as cleanup:
            for document_id in created_ids:
                await delete_document(cleanup, owner, document_id)
        print(f"Cleaned up {len(created_ids)} notebook documents.")
    finally:
        await embedder.aclose()
        await engine.dispose()


=== Document parser messages ===



=== Document parser messages ===



=== Document parser messages ===


Skipped scanned_office_notice.pdf: no text layer.



=== Document parser messages ===


Indexed 18 documents with bge-m3.



Question: How many annual leave days do employees get?
 dense | No. | Similarity | Keyword rank | RRF score | Source
 dense |   1 |     0.6562 |            - |  0.000000 | leave_policy_v2_2026.pdf (page 1 / Tarnwick Leave Policy)
 dense |   2 |     0.6367 |            - |  0.000000 | employee_faq.txt (page - / -)
 dense |   3 |     0.6195 |            - |  0.000000 | leave_policy_v1_2024.docx (page - / Tarnwick Leave Policy)
 dense |   4 |     0.6142 |            - |  0.000000 | leave_policy_v1_2024.docx (page - / 4. Requesting Leave)
hybrid | No. | Similarity | Keyword rank | RRF score | Source
hybrid |   1 |     0.6367 |            1 |  0.032522 | employee_faq.txt (page - / -)
hybrid |   2 |     0.6562 |            - |  0.016393 | leave_policy_v2_2026.pdf (page 1 / Tarnwick Leave Policy)
hybrid |   3 |     0.6195 |            - |  0.015873 | leave_policy_v1_2024.docx (page - / Tarnwick Leave Policy)
hybrid |   4 |     0.6142 |            - |  0.015625 | leave_policy_v1_2024.docx (pa

## Interpret cosine similarity

Cosine similarity measures the relationship between the query and the retrieved text. Hybrid ranking can move a keyword match up while preserving its cosine score. Compare the off-topic score with the relevant hits before tuning the minimum similarity, then check the choice over a larger evaluation set.